In [1]:
import os
import numpy as np
import torch
from argparse import ArgumentParser
import tqdm 

from config import *
from helpers import * 
import articulate as art
from constants import MODULES
from utils.model_utils import load_model
from data import PoseDataset
from models import MobilePoserNet

import argparse
import os
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset



# dataset 

In [2]:
from data import PoseDataset
import torch

ds = PoseDataset(fold='train', finetune=None)   # loads processed_datasets
pairs = []   # list of (pose_seq, tran_seq) per window/sequence
for i in range(len(ds)):
    item = ds[i]
    # training dataset returns (imu, pose, joint, tran, vel, contact)
    imu, pose, joint, tran = item[:4]
    # pose: (T, 6 * num_pred_joints) , tran: (T, 3)
    pairs.append((pose, tran))


# # example: concatenate all sequences (if lengths match) or keep as list
all_poses = torch.cat([p for p, t in pairs], dim=0)
all_trans  = torch.cat([t for p, t in pairs], dim=0)

device = "cpu"
x0 = torch.cat([all_poses.to(device), all_trans.to(device)], dim=-1)  # (B, 147)


Loading datasets for TRAIN mode (finetune=None, evaluate=None)
Datasets: ['ACCAD.pt', 'BioMotionLab_NTroje.pt', 'BMLhandball.pt', 'BMLmovi.pt', 'CMU.pt', 'DanceDB.pt', 'DFaust_67.pt', 'dip_test.pt', 'dip_train.pt', 'Eyes_Japan_Dataset.pt', 'HUMAN4D.pt', 'HumanEva.pt', 'MPI_HDM05.pt', 'MPI_Limits.pt', 'MPI_mosh.pt', 'SFU.pt', 'SSM_synced.pt', 'TCD_handMocap.pt', 'TotalCapture.pt', 'Transitions_mocap.pt']



  0%|          | 0/20 [00:00<?, ?it/s]c:\deepika\2_mobileposer\MobilePoser\mobileposer\data.py:55: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  file_data = torch.load(data_

# model 

In [3]:
import torch
import torch.nn as nn
import math

class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        emb = math.log(10000) / (half - 1)
        emb = torch.exp(torch.arange(half, device=t.device) * -emb)
        emb = t[:, None] * emb[None, :]
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        return emb


class DiffusionMLP(nn.Module):
    def __init__(self, dim=147, cond_dim=147, hidden=1024):
        super().__init__()
        
        self.time_embed = TimeEmbedding(128)
        
        self.net = nn.Sequential(
            nn.Linear(dim + cond_dim + 128, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, dim)
        )

    def forward(self, x_t, t, cond):
        t_emb = self.time_embed(t)
        h = torch.cat([x_t, cond, t_emb], dim=-1)
        return self.net(h)

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

# Example dataset
class PoseDataset(Dataset):
    def __init__(self, x, cond):
        # Ensure they are tensors
        self.x = x if torch.is_tensor(x) else torch.tensor(x, dtype=torch.float32)
        self.cond = cond if torch.is_tensor(cond) else torch.tensor(cond, dtype=torch.float32)

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        # idx could be a tensor if using batch sampler, convert to int
        if torch.is_tensor(idx):
            idx = idx.item()
        # print(self.x[idx].shape)
        return self.x[idx], self.cond[idx]


device = "cpu"
batch_size = 32
learning_rate = 1e-4
num_epochs = 2
timesteps = 1000  # Number of diffusion steps


cond = torch.zeros_like(x0)  # (batch_size, 147)
dataset = PoseDataset(x0, cond)

dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Initialize model
model = DiffusionMLP(dim=147, cond_dim=147, hidden=1024).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = nn.MSELoss()

# Linear noise schedule (beta)
beta_start, beta_end = 1e-4, 0.02
beta = torch.linspace(beta_start, beta_end, timesteps).to(device)
alpha = 1 - beta
alpha_bar = torch.cumprod(alpha, dim=0)



# testing 

In [5]:
from articulate import evaluator
# 1. Initialize your evaluator
evaluator_ = evaluator.MeanPerJointErrorEvaluator(
    official_model_file=r"C:\deepika\2_mobileposer\MobilePoser\mobileposer\smpl\basicmodel_m.pkl",
    rep=evaluator.RotationRepresentation.R6D, # Check if your evaluator supports R6D directly
    device=device
)


In [ ]:
import torch
from tqdm.auto import tqdm

@torch.no_grad()
def test_model(model, all_poses_gt, all_trans_gt, timesteps, alpha, alpha_bar, beta, evaluator_):
    """
    Tests the diffusion model using concatenated ground truth tensors.
    """
    # Load the best weights saved during training
    model.load_state_dict(torch.load(r'C:\deepika\2_mobileposer\MobilePoser\mobileposer\model_best.pt', weights_only=True))
    model.eval()
    
    # 1. Initialize x_t as pure Gaussian noise (B, 147)
    # Using a subset (e.g., first 32) if memory is an issue, otherwise all_poses_gt.shape[0]
    batch_size = all_poses_gt.shape[0]
    x = torch.randn((batch_size, 147)).to(device)
    
    
    # 2. Iterate backwards from T to 1 (Denoising process)
    for i in tqdm(reversed(range(timesteps)), total=timesteps, desc="Denoising"):
        t = torch.full((batch_size,), i, device=device, dtype=torch.long)
        
        # Predict noise using current condition (must match training: currently zeros)
        cond = torch.zeros_like(x) 
        noise_pred = model(x, t.float(), cond)
        
        # DDPM Denoising Step Constants
        a = alpha[i]
        ab = alpha_bar[i]
        b = beta[i]
        
        z = torch.randn_like(x) if i > 0 else 0
            
        # Refine x: move from noisy state x_t to less noisy state x_{t-1}
        x = (1 / torch.sqrt(a)) * (x - ((1 - a) / (torch.sqrt(1 - ab))) * noise_pred) + torch.sqrt(b) * z

    # 3. Post-Process for Evaluator
    # Split the 147-dim vector back into Pose (144) and Tran (3)
    pred_pose_r6d = x[:, :144].reshape(batch_size, 24, 6).contiguous() # Predicted R6D pose
    pred_tran = x[:, 144:].contiguous()                               # Predicted translation
    
    # Ground truth reshape for the evaluator
    gt_pose_r6d = all_poses_gt.reshape(batch_size, 24, 6).contiguous()
    
    # 4. Evaluation
    # Calculates Mean Position Error (MPJPE), Local Rotation, and Global Rotation
    error_array = evaluator_(pred_pose_r6d, gt_pose_r6d)
    
    return error_array, pred_pose_r6d, pred_tran

# --- How to run the evaluation ---
# Use the all_poses and all_trans you created from the dataset loop
# Taking the first 64 samples as a test batch
batch_pose_gt = all_poses[:256].to(device)
batch_tran_gt = all_trans[:256].to(device)

results = test_model(model, batch_pose_gt, batch_tran_gt, timesteps, alpha, alpha_bar, beta, evaluator_)
print(f"MPJPE: {results[0][0].item()} | Local Rot: {results[0][1].item()} | Global Rot: {results[0][2].item()}")

Denoising:   0%|          | 0/1000 [00:00<?, ?it/s]

MPJPE: 0.23157882690429688 | Local Rot: 105.13074493408203 | Global Rot: 99.78668975830078
